# Домашнее задание блока 3. Лекция 5

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/homeworks/hw3/homework.ipynb)

Задание по лекции 5: стратегия с параметрами, REINFORCE и вычитание среднего. В первой части — коридор, где агент не различает клеток: лучшую стратегию вы посчитаете точно и проверите, что REINFORCE к ней приходит. Во второй — стратегия-нейросеть на новой среде Acrobot. Берите за основу функции семинара 5 и код раздела 5 лекции: задание на это рассчитано.

Всего **100 баллов** и бонус:

| Часть | Среда | Что проверяем | Баллы |
|---|---|---|---|
| 1. Коридор с серыми клетками | коридор из шести клеток | лучшая стратегия: без случайности там, где агент различает клетки, и со случайностью там, где не различает; REINFORCE её находит | 50 |
| 2. Acrobot: стратегия-нейросеть | Acrobot-v1 | все награды −1: без вычитания среднего обучение медленнее и разброс между запусками больше | 50 |
| ★ Бонус. Cross-Entropy как вес 1/0 | Acrobot-v1 | Cross-Entropy — та же оценка, где вместо $G_t - b$ стоит 1 у элитных эпизодов и 0 у остальных | +10 |

Каждое задание разбито на пронумерованные пункты, и у каждого пункта своя заготовка `TODO` в коде. Части с `assert` проверяются автоматически при запуске ячейки: если `assert` не упал, часть засчитана. Графики и письменные ответы проверяются вручную. Пишите их прямо в markdown-ячейках вместо слова `TODO`.

Целиком ноутбук считается около 3 минут (вместе с бонусной частью). Задание рассчитано на 2–3 часа.

**Не забудьте про квиз.** К этому блоку он один, по лекции 5. Пройдите его в зачётном режиме и положите код результата в `quiz/week05.txt`.

**Дедлайн:** преподаватель объявит при выдаче задания.

Первые две ячейки служебные: установка пакетов в Colab и подключение модуля с картинками. Запустите и идите дальше.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
for package, pip_name in (("gymnasium", "gymnasium[classic-control]"), ("torch", "torch")):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

In [ ]:
# Служебное: модуль с коридором и картинками. Читать не нужно, сигнатуры — в его docstring.
import pathlib, sys, urllib.request

name, folder = "hw3_plots", "homeworks/hw3"
local = next((p for p in ["", f"../../{folder}", f"../../../{folder}", folder] if (pathlib.Path(p) / f"{name}.py").exists()), None)
if local is None:                                                  # в Colab скачиваем из репозитория курса
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/IlyaChichkanov/Reinforcement-learning/main/{folder}/{name}.py", f"{name}.py")
    local = ""
sys.path.insert(0, str(pathlib.Path(local).resolve()))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
from IPython.display import display
from hw3_plots import (CELLS, EXIT, STARTS, LEFT, RIGHT, OBS_NAMES, observe, next_cell, GreyCorridor,
                       draw_grey_corridor, plot_value_curves, plot_prob_runs,
                       draw_acrobot, plot_lengths, animate_acrobot)

## Часть 1. Коридор с серыми клетками (50 баллов)

*Это коридор из раздела 2 лекции, только длиннее и с выходом посередине.* Среда по чек-листу:

* **Клетки** — шесть, с номерами 0…5; клетка 2 — выход.
* **Действия** — два: ← (номер 0) и → (номер 1). Шаг в стену (← в клетке 0 или → в клетке 5) оставляет агента на месте.
* **Награда** — −1 за каждый шаг; $\gamma = 1$, поэтому return эпизода — минус число шагов до выхода.
* **Старт** — одна из пяти клеток 0, 1, 3, 4, 5, все равновероятны. **Конец** — выход или 200 шагов.
* **Что видит агент.** Датчик различает только три случая: «стена слева» (клетка 0), «стена справа» (клетка 5) и «серая клетка» — клетки 1, 3 и 4 для датчика одинаковы.

Поэтому стратегия — три числа: вероятность → при каждом наблюдении, $p_L$ («стена слева»), $p_G$ («серая клетка») и $p_R$ («стена справа»). В серых клетках правильные действия разные: из клетки 1 к выходу ведёт →, из клеток 3 и 4 — ←.

In [ ]:
draw_grey_corridor(title="коридор: сверху номера клеток, снизу — что видит датчик")
plt.show()

env = GreyCorridor()
obs, info = env.reset(seed=0)
print(f"старт в клетке {info['cell']}, датчик видит: {OBS_NAMES[obs]}")
obs, r, terminated, truncated, info = env.step(LEFT)
print(f"после ←: клетка {info['cell']}, награда {r}, выход: {terminated}")

### 1.1 Точная ценность старта (20 баллов)

Для заданной стратегии ценность каждой клетки удовлетворяет уравнению Беллмана для стратегии из лекции 2, как три уравнения коридора в разделе 2 лекции 5. В клетке $s$ датчик видит $o(s)$, и вероятность → там равна $p_{o(s)}$:

$$
V(s) \;=\; -1 \;+\; p_{o(s)}\, V(s_{\rightarrow}) \;+\; \big(1 - p_{o(s)}\big)\, V(s_{\leftarrow}) , \qquad V(2) = 0 ,
$$

где $s_{\rightarrow}$ и $s_{\leftarrow}$ — клетки, куда ведут → и ← (их даёт `next_cell(s, a)`), а $V(2) = 0$ — ценность выхода. Неизвестных пять: $V(0), V(1), V(3), V(4), V(5)$; уравнений тоже пять, по одному на клетку:

$$
\begin{aligned}
V(0) &= -1 + p_L\, V(1) + (1 - p_L)\, V(0)\\
V(1) &= -1 + p_G \cdot 0 + (1 - p_G)\, V(0)\\
V(3) &= -1 + p_G\, V(4) + (1 - p_G) \cdot 0\\
V(4) &= -1 + p_G\, V(5) + (1 - p_G)\, V(3)\\
V(5) &= -1 + p_R\, V(5) + (1 - p_R)\, V(4)
\end{aligned}
$$

В клетке 0 действие ← упирается в стену, и агент остаётся в клетке 0, поэтому в правой части первого уравнения стоит $V(0)$. С → в клетке 5 то же самое. Из клетки 1 действие → и из клетки 3 действие ← ведут в выход, где ценность 0.

Перенесём все $V$ в левую часть и запишем систему как $A\,\vec V = -\vec 1$ (справа в каждой строке $-1$). В строке клетки $s$ на диагонали стоит 1, в столбце клетки $s_{\rightarrow}$ — число $-p_{o(s)}$, в столбце клетки $s_{\leftarrow}$ — число $-(1 - p_{o(s)})$. Если $s_{\rightarrow}$ или $s_{\leftarrow}$ — выход, слагаемого нет, потому что $V(2) = 0$; если это сама клетка $s$, число вычитается из диагонали.

**Ценность старта** $J$ — среднее $V$ по пяти стартовым клеткам.

**Проверка руками.** При $p_L = 1$, $p_G = 0.5$, $p_R = 0$: из клетки 1 половина шагов ведёт сразу к выходу, а половина — в клетку 0 и обратно, отсюда $V(1) = -1 + 0.5\,(-1 + V(1))$, то есть $V(1) = -3$ и $V(0) = -4$. Так же получаются $V(3) = -5$, $V(4) = -8$, $V(5) = -9$ и $J = (-4 - 3 - 5 - 8 - 9)/5 = -5.8$.

1. **TODO 1.1 (1).** Напишите `start_value(pL, pG, pR)`: соберите $A$ и правую часть, решите систему `np.linalg.solve` и верните $J$.
2. **TODO 1.1 (2).** Постройте ценность старта против $p_G$ на сетке `ps` для двух стратегий в клетках 0 и 5: (а) $p_L = 1$, $p_R = 0$: там без случайности, к выходу; (б) $p_L = p_R = 0.5$: там → и ← поровну. Найдите лучшее $p_G$ для (а) и запишите его в `BEST_P`, а ценность — в `BEST_J`.

In [ ]:
WAYS = [c for c in range(CELLS) if c != EXIT]          # клетки с неизвестной ценностью: 0, 1, 3, 4, 5

def start_value(pL, pG, pR):
    """Ценность старта J — среднее V по стартовым клеткам — для вероятностей → (pL, pG, pR)."""
    p = (pL, pG, pR)                                     # p[observe(cell)] — вероятность → в клетке
    # TODO 1.1 (1): матрица A (5 × 5) и правая часть из −1, решение np.linalg.solve, среднее по STARTS
    raise NotImplementedError

In [ ]:
# Самопроверка 1.1 (1) — пример из текста
J_check = start_value(1.0, 0.5, 0.0)
assert np.isclose(J_check, -5.8), f"при p = (1, 0.5, 0) ценность старта −5.8 (V = −4, −3, −5, −8, −9), у вас {J_check:.3f}"
assert start_value(1.0, 0.4, 0.0) > start_value(1.0, 0.9, 0.0), "чаще → в серых клетках — дольше бродить справа от выхода"
print("OK: J(1, 0.5, 0) =", round(J_check, 3))

In [ ]:
ps = np.linspace(0.02, 0.98, 97)
# TODO 1.1 (2): ценность старта против p_G для двух стратегий в клетках 0 и 5; лучшее p_G для (а) — в BEST_P и BEST_J;
#               картинка: plot_value_curves(ps, {подпись: значения, ...}, best=(BEST_P, BEST_J))
BEST_P, BEST_J = None, None

In [ ]:
# Самопроверка 1.1 (2)
assert BEST_P is not None and 0.28 < BEST_P < 0.38, f"лучшее p_G — около 1/3, у вас {BEST_P}"
assert np.isclose(BEST_J, start_value(1.0, BEST_P, 0.0)), "BEST_J — ценность старта при BEST_P для стратегии (а)"
print(f"OK: лучшее p_G = {BEST_P:.2f}, ценность {BEST_J:.2f}")

**Вопросы к 1.1.** (а) Что происходит с ценностью старта при $p_G \to 0$ и при $p_G \to 1$ и почему? Проследите, куда ходит агент. (б) Почему лучшее $p_G$ меньше 0.5? (в) Почему в клетках 0 и 5 случайность только вредит, а в серых клетках без неё нельзя?

_Ваш ответ:_ TODO

### 1.2 REINFORCE с тремя предпочтениями (30 баллов)

Стратегия — три предпочтения $\vec\theta = (\theta_L, \theta_G, \theta_R)$, по одному на каждое наблюдение, а вероятность → при наблюдении $o$ считается по формуле раздела 3 лекции: $p_o = e^{\theta_o} / (1 + e^{\theta_o})$. Шаг в эпизоде меняет только предпочтение того наблюдения, которое агент видел: производная $\log \pi(a \mid o)$ по $\theta_o$ равна $[a = {\rightarrow}] - p_o$, по двум другим предпочтениям — нулю.

Шаг REINFORCE с вычитанием среднего (разделы 4–5 лекции): после эпизода

$$
\vec\theta \;\leftarrow\; \vec\theta + \alpha \sum_{t=0}^{T-1} \gamma^t\, (G_t - b)\, \nabla_{\vec\theta} \log \pi(a_t \mid o_t) ,
$$

где $G_t = \sum_{k \ge t} \gamma^{k-t} r_k$ — сумма наград с шага $t$ до конца эпизода, а $b$ — средний $G_t$ всех прошлых шагов. В коридоре $\gamma = 1$, поэтому $\gamma^t = 1$, а $G_t$ — минус число шагов, оставшихся до выхода. Слагаемое шага $t$ добавляет $(G_t - b)\,([a_t = {\rightarrow}] - p_{o_t})$ к компоненте $o_t$.

**Проверка руками.** Эпизод из клетки 0: → (датчик: «стена слева», $o = 0$), затем → из клетки 1 (датчик: «серая», $o = 1$), и агент на выходе. Награды $-1, -1$, поэтому $G_0 = -2$, $G_1 = -1$. При $\vec\theta = 0$ все $p = 0.5$, и при $b = 0$ оценка равна $\big(-2 \cdot 0.5,\ -1 \cdot 0.5,\ 0\big) = (-1, -0.5, 0)$; при $b = -1.5$ — $\big(-0.5 \cdot 0.5,\ 0.5 \cdot 0.5,\ 0\big) = (-0.25, 0.25, 0)$.

1. **TODO 1.2 (1).** Напишите `grad_estimate(theta, observations, actions, rewards, b)`.
2. **TODO 1.2 (2).** Допишите в каркас `reinforce` одну строку с шагом по `grad_estimate` и обучите 10 запусков по 3000 эпизодов с $\alpha = 0.01$. Нарисуйте $p_G$ по эпизодам вместе с точным ответом `BEST_P`, а рядом $p_L$ и $p_R$.

In [ ]:
def p_right(theta):
    """Вероятность → при предпочтении θ (предпочтение ← равно 0); θ может быть массивом."""
    return np.exp(theta) / (1 + np.exp(theta))

def grad_estimate(theta, observations, actions, rewards, b=0.0):
    """Σ_t (G_t − b) ∇ log π(a_t | o_t) — вектор из трёх чисел."""
    # TODO 1.2 (1): G_t — суммы наград с конца; к компоненте observations[t] прибавить (G_t − b) · ([a_t = →] − p)
    raise NotImplementedError

In [ ]:
# Самопроверка 1.2 (1) — эпизод из текста
toy = dict(observations=[0, 1], actions=[RIGHT, RIGHT], rewards=np.array([-1.0, -1.0]))
g0 = grad_estimate(np.zeros(3), **toy)
assert np.allclose(g0, [-1, -0.5, 0]), f"при b = 0 оценка (−1, −0.5, 0), у вас {g0}"
g1 = grad_estimate(np.zeros(3), **toy, b=-1.5)
assert np.allclose(g1, [-0.25, 0.25, 0]), f"при b = −1.5 оценка (−0.25, 0.25, 0), у вас {g1}"
print("OK:", g0, g1)

In [ ]:
def play(env, theta, rng):
    """Один эпизод стратегией с предпочтениями theta; возвращает наблюдения, действия и награды."""
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    observations, actions, rewards = [], [], []
    while True:
        a = int(rng.random() < p_right(theta[obs]))
        observations.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        rewards.append(r)
        if terminated or truncated:
            return observations, actions, np.array(rewards)

def reinforce(alpha=0.01, n_episodes=3000, seed=0):
    """REINFORCE с вычитанием среднего; возвращает вероятности → (p_L, p_G, p_R) после каждого эпизода."""
    rng = np.random.default_rng(seed)
    env, theta = GreyCorridor(), np.zeros(3)
    b, n, history = 0.0, 0, []
    for _ in range(n_episodes):
        observations, actions, rewards = play(env, theta, rng)
        # TODO 1.2 (2): одна строка — шаг θ ← θ + α · grad_estimate(...) со средним b
        raise NotImplementedError
        for g in np.cumsum(rewards[::-1])[::-1]:       # b — средний G_t всех прошлых шагов
            n += 1; b += (g - b) / n
        history.append(p_right(theta))
    return np.array(history)

# TODO 1.2 (2): 10 запусков (seed 0…9), массив runs формы (10, 3000, 3); три картинки plot_prob_runs: p_G с target=BEST_P, p_L, p_R
runs = None

In [ ]:
# Самопроверка 1.2 (2)
assert runs is not None and runs.shape == (10, 3000, 3), "runs — массив (10 запусков, 3000 эпизодов, 3 вероятности)"
final = runs[:, -1]
assert abs(np.median(final[:, 1]) - BEST_P) < 0.08, \
    f"медиана p_G в конце должна быть близко к точному {BEST_P:.2f}, у вас {np.median(final[:, 1]):.2f}"
assert np.sum((final[:, 0] > 0.9) & (final[:, 2] < 0.15)) >= 8, \
    "в клетках 0 и 5 большинство запусков должно прийти к стратегии без случайности: p_L > 0.9, p_R < 0.15"
print("OK: медиана p_G =", round(np.median(final[:, 1]), 2))

**Вопросы к 1.2.** (а) Куда пришли запуски по $p_G$ и насколько они разбросаны вокруг точного ответа? Объясните разброс по форме кривой (а) из 1.1 около её вершины. (б) Q-learning по таблице выбирает в каждом наблюдении одно действие с наибольшим $Q$. Что он выберет бы в серых клетках и чем это кончится?

_Ваш ответ:_ TODO

## Часть 2. Acrobot: стратегия-нейросеть (50 баллов)

*Это раздел 5 лекции на новой среде.* Среда `Acrobot-v1` по чек-листу:

* **Устройство** — два звена на шарнирах: верхний шарнир закреплён, в среднем стоит мотор.
* **Состояние** — 6 чисел: $\cos$ и $\sin$ угла первого звена, $\cos$ и $\sin$ угла второго звена относительно первого и две угловые скорости.
* **Действия** — три: момент мотора $-1$, $0$ или $+1$ (номера 0, 1, 2).
* **Награда** — −1 за каждый шаг, пока конец нижнего звена не поднимется выше пунктира: высота 1 над закреплённым шарниром, при длине звена 1.
* **Конец эпизода** — конец звена выше пунктира или 500 шагов. Поэтому return — минус число шагов, а мерка всей части — **шагов до подъёма**: чем меньше, тем лучше; 500 значит «не поднял».

Стратегия — нейросеть 6 → 32 → 3: на вход подаём состояние, на выходе получаем три предпочтения, а вероятности из них даёт softmax, как в разделе 3 лекции, только в знаменателе стоит сумма $e^\theta$ по трём действиям. Производную $\log \pi$ по весам сети считает PyTorch. Суммы наград $G_t = r_t + \gamma\, G_{t+1}$ считает готовая функция `returns_to_go`: с конца эпизода, как на семинаре 5.

In [ ]:
acro = gym.make("Acrobot-v1")
s0, _ = acro.reset(seed=0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
draw_acrobot(s0, ax=axes[0], title="старт:\nзвенья висят вниз")
draw_acrobot([np.cos(2.6), np.sin(2.6), np.cos(-0.9), np.sin(-0.9), 0, 0], ax=axes[1], title="конец звена выше пунктира:\nэпизод окончен")
plt.show()

def make_policy(seed):
    """Стратегия-нейросеть 6 → 32 → 3 с заданным начальным seed."""
    torch.manual_seed(seed)
    return torch.nn.Sequential(torch.nn.Linear(6, 32), torch.nn.Tanh(), torch.nn.Linear(32, 3))

def returns_to_go(rewards, gamma):
    """G_t = r_t + γ G_{t+1} для каждого шага эпизода, считаем с конца."""
    G, acc = np.zeros(len(rewards)), 0.0
    for t in reversed(range(len(rewards))):
        acc = rewards[t] + gamma * acc
        G[t] = acc
    return G

print("G_t для наград (−1, −1, −1) при γ = 0.5:", returns_to_go([-1.0, -1.0, -1.0], 0.5))

### 2.1 Эпизод и функция потерь (20 баллов)

1. **TODO 2.1 (1).** Допишите `play_net`: в каждом состоянии посчитайте вероятности трёх действий (softmax выхода сети), выберите действие случайно по ним (`rng.choice(3, p=probs / probs.sum())`: деление убирает ошибку округления, из-за которой сумма вероятностей бывает не ровно 1), сделайте шаг среды и запишите состояние, действие и награду.
2. **TODO 2.1 (2).** Напишите `reinforce_loss(net, states, actions, rewards, gamma, b)`, функцию потерь из раздела 5 лекции. Её градиент равен минус оценке REINFORCE:
$$
\text{loss}(\theta) \;=\; -\sum_{t=0}^{T-1} \gamma^t\, (G_t - b)\, \log \pi_\theta(a_t \mid s_t) .
$$
   `log_softmax` выхода сети даёт $\log \pi$ всех действий, из каждой строки берите столбец сыгранного действия.

**Проверка руками для 2.1 (2).** У сети с нулевыми весами все три вероятности равны $1/3$, и $\log \pi = -\log 3$. Эпизод из двух шагов с наградами $-1, -1$ и $\gamma = 0.5$: $G_0 = -1.5$, $G_1 = -1$. При $b = 0$ веса шагов $\gamma^t G_t$ равны $-1.5$ и $-0.5$, и $\text{loss} = -(-1.5 - 0.5)(-\log 3) = -2 \log 3 \approx -2.197$; при $b = -1$ веса $-0.5$ и $0$, $\text{loss} = -0.5 \log 3 \approx -0.549$.

In [ ]:
def play_net(env, net, rng, seed=None):
    """Один эпизод стратегией-сетью; возвращает списки состояний, действий и наград."""
    s, _ = env.reset(seed=seed)
    states, actions, rewards = [], [], []
    while True:
        # TODO 2.1 (1): probs = softmax выхода сети (под torch.no_grad(), .numpy()); a = rng.choice(3, p=probs / probs.sum());
        #               шаг среды; записать s, a, r
        raise NotImplementedError
        if terminated or truncated:
            return states, actions, rewards

def reinforce_loss(net, states, actions, rewards, gamma=0.99, b=0.0):
    """loss = −Σ_t γ^t (G_t − b) log π(a_t | s_t); тензор из одного числа."""
    # TODO 2.1 (2): G_t — returns_to_go(rewards, gamma); log_softmax выхода сети для всех состояний;
    #               столбцы сыгранных действий; loss
    raise NotImplementedError

In [ ]:
# Самопроверка 2.1
class Fixed(torch.nn.Module):                      # «сеть» с заданными предпочтениями, одинаковыми в любом состоянии
    def __init__(self, prefs): super().__init__(); self.prefs = torch.tensor(prefs)
    def forward(self, x): return self.prefs.expand(*x.shape[:-1], 3)

rng = np.random.default_rng(0)
_, acts, rews = play_net(acro, Fixed([0.0, 0.0, 20.0]), rng, seed=0)
assert set(acts) == {2} and all(r == -1.0 for r in rews), "при предпочтениях (0, 0, 20) действие всегда 2, награды −1"
_, acts, _ = play_net(acro, Fixed([0.0, 0.0, 0.0]), rng, seed=0)
assert set(acts) == {0, 1, 2}, "при равных предпочтениях выпадают все три действия"
zero = torch.nn.Sequential(torch.nn.Linear(6, 3)); torch.nn.init.zeros_(zero[0].weight); torch.nn.init.zeros_(zero[0].bias)
toy = dict(states=[np.zeros(6), np.zeros(6)], actions=[0, 2], rewards=[-1.0, -1.0], gamma=0.5)
assert np.isclose(reinforce_loss(zero, **toy).item(), -2 * np.log(3), atol=1e-5), "при b = 0 loss = −2 log 3 ≈ −2.197"
assert np.isclose(reinforce_loss(zero, **toy, b=-1.0).item(), -0.5 * np.log(3), atol=1e-5), "при b = −1 loss = −0.5 log 3 ≈ −0.549"
lengths_random = [len(play_net(acro, make_policy(k), rng, seed=k)[2]) for k in range(10)]
print(f"OK: необученная сеть поднимает конец в среднем за {np.mean(lengths_random):.0f} шагов (лимит 500)")

### 2.2 Обучение с вычитанием среднего и без (20 баллов)

Каркас `train` готов: он играет эпизоды, после каждого делает шаг Adam по `reinforce_loss` и записывает, сколько шагов понадобилось на подъём. Шаг $\alpha = $ `lr` $= 0.005$, $\gamma = 0.99$, по 300 эпизодов.

1. **TODO 2.2 (1).** Допишите в `train` одну строку: `loss` по `reinforce_loss` с $b$, если `baseline=True`, и с нулём иначе.
2. **TODO 2.2 (2).** Обучите по 5 запусков (seed 0…4) без вычитания среднего и с ним (отличается только `baseline`) и нарисуйте `plot_lengths` для обоих вариантов на двух панелях. Мерка запуска — средняя длина его последних 30 эпизодов. Счёт займёт около 2 минут: 10 запусков по 300 эпизодов.

In [ ]:
def train(seed, baseline, n_episodes=300, lr=0.005, gamma=0.99):
    """REINFORCE для стратегии-сети; возвращает длины эпизодов и сеть."""
    rng, net = np.random.default_rng(seed), make_policy(seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    env = gym.make("Acrobot-v1")
    b, n, lengths = 0.0, 0, []
    for episode in range(n_episodes):
        states, actions, rewards = play_net(env, net, rng, seed=seed * 10_000 + episode)
        # TODO 2.2 (1): одна строка — loss = reinforce_loss(...) с b при baseline=True и с 0.0 иначе
        raise NotImplementedError
        opt.zero_grad(); loss.backward(); opt.step()
        for g in returns_to_go(rewards, gamma):        # b — средний G_t всех прошлых шагов
            n += 1; b += (g - b) / n
        lengths.append(len(rewards))
    return np.array(lengths), net

# TODO 2.2 (2): по 5 запусков без вычитания и с ним — массивы длин lengths_plain, lengths_base формы (5, 300),
#               сети запусков с вычитанием — в список nets_base; две панели plot_lengths
lengths_plain, lengths_base, nets_base = None, None, None

In [ ]:
# Самопроверка 2.2
assert lengths_base is not None and lengths_base.shape == (5, 300) and lengths_plain.shape == (5, 300), \
    "lengths_plain и lengths_base — массивы (5 запусков, 300 эпизодов)"
last_base, last_plain = lengths_base[:, -30:].mean(axis=1), lengths_plain[:, -30:].mean(axis=1)
assert np.sum(last_base < 200) >= 2, f"с вычитанием среднего хотя бы два запуска из пяти должны поднимать конец быстрее 200 шагов: {last_base.round(0)}"
print(f"OK: медиана последних 30 эпизодов — без вычитания {np.median(last_plain):.0f}, с вычитанием {np.median(last_base):.0f} шагов")

**Вопросы к 2.2.** (а) Сравните кривые: какой вариант учится быстрее и у какого меньше разброс между запусками? Объясните механизм по аналогии с CartPole из лекции: какой знак у всех $G_t$ в Acrobot и что без вычитания среднего шаг делает с вероятностью каждого сыгранного действия? (б) Если какой-то запуск так и остался у лимита 500, посмотрите на его длины эпизодов (`lengths_base[k]`): поднял ли он конец хоть раз? Чем тогда эпизоды отличаются друг от друга по наградам и чему может научиться шаг REINFORCE?

_Ваш ответ:_ TODO

### 2.3 Как поднимает конец обученная сеть (10 баллов)

**TODO 2.3.** Возьмите сеть лучшего запуска с вычитанием среднего и сыграйте эпизод без случайности: в каждом состоянии берите действие с наибольшей вероятностью. Соберите все состояния в `states_best`, а действия — в `actions_best`; `animate_acrobot` сделает из них GIF.

In [ ]:
best = int(np.argmin(lengths_base[:, -30:].mean(axis=1)))
net_best = nets_base[best]
s, _ = acro.reset(seed=123)
states_best, actions_best = [s], []
# TODO 2.3: цикл — действие с наибольшим выходом сети (под torch.no_grad()), шаг среды, запись состояния и действия;
#           выход из цикла при terminated или truncated

print("шагов до подъёма:", len(actions_best))
display(animate_acrobot(states_best, actions_best))

In [ ]:
# Самопроверка 2.3
assert len(actions_best) > 0 and len(states_best) == len(actions_best) + 1, "в states_best на одно состояние больше, чем действий"
assert len(actions_best) < 500, "обученная сеть должна поднять конец звена раньше лимита в 500 шагов"
print("OK")

## ★ Бонус. Cross-Entropy как вес 1/0 (+10 баллов)

В разделе 4 лекции Cross-Entropy был той же оценкой градиента, только вместо return стоял вес $w(\tau)$: 1 у элитного эпизода и 0 у остального. Проверим это на Acrobot. Элитный эпизод — тот, где на подъём ушло не больше шагов, чем у 30 % самых коротких из последних 20 эпизодов (порог `np.quantile(recent, 0.3)`). В первых 5 эпизодах шага нет: сравнивать пока не с чем. Тогда функция потерь

$$
\text{loss}_{\text{CE}}(\theta) \;=\; -\, w(\tau) \sum_{t=0}^{T-1} \log \pi_\theta(a_t \mid s_t) .
$$

**TODO ★.** Допишите `ce_loss` и обучите 5 запусков тем же каркасом: `train_ce` уже передаёт в неё всё нужное. Нарисуйте кривые рядом с REINFORCE с вычитанием среднего из 2.2 и ответьте: чем шаг Cross-Entropy отличается от шага REINFORCE с вычитанием среднего, если смотреть на неудачные эпизоды?

In [ ]:
def ce_loss(net, states, actions, elite):
    """loss_CE = −w Σ_t log π(a_t | s_t), где w = 1 у элитного эпизода и 0 у остального."""
    # TODO ★: log_softmax выхода сети, столбцы сыгранных действий, сумма с весом 1.0 или 0.0
    raise NotImplementedError

def train_ce(seed, n_episodes=300, lr=0.005):
    """Тот же каркас, что train из 2.2, но шаг — по ce_loss; возвращает длины эпизодов."""
    rng, net = np.random.default_rng(seed), make_policy(seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    env = gym.make("Acrobot-v1")
    lengths = []
    for episode in range(n_episodes):
        states, actions, rewards = play_net(env, net, rng, seed=seed * 10_000 + episode)
        recent = lengths[-20:]
        elite = len(recent) >= 5 and len(rewards) <= np.quantile(recent, 0.3)
        loss = ce_loss(net, states, actions, elite)
        opt.zero_grad(); loss.backward(); opt.step()
        lengths.append(len(rewards))
    return np.array(lengths)

# TODO ★: 5 запусков train_ce — массив lengths_ce формы (5, 300); две панели plot_lengths: REINFORCE с вычитанием и Cross-Entropy
lengths_ce = None

In [ ]:
# Самопроверка ★
assert lengths_ce is not None and lengths_ce.shape == (5, 300), "lengths_ce — массив (5 запусков, 300 эпизодов)"
last_ce = lengths_ce[:, -30:].mean(axis=1)
assert np.sum(last_ce < 200) >= 2, f"Cross-Entropy тоже должен учиться: хотя бы два запуска из пяти быстрее 200 шагов, у вас {last_ce.round(0)}"
print(f"OK: медиана последних 30 эпизодов — Cross-Entropy {np.median(last_ce):.0f}, REINFORCE с вычитанием {np.median(last_base):.0f} шагов")

_Ваш ответ:_ TODO

## Чек-лист перед сдачей

- [ ] 1.1: `start_value` проходит проверку −5.8; кривые ценности против $p_G$, `BEST_P` и `BEST_J`; ответы на вопросы
- [ ] 1.2: `grad_estimate` проходит проверку на эпизоде из текста; 10 запусков, три картинки; ответы на вопросы
- [ ] 2.1: `play_net` и `reinforce_loss` проходят самопроверку
- [ ] 2.2: по 5 запусков без вычитания среднего и с ним, две панели кривых; ответ
- [ ] 2.3: эпизод обученной сети и GIF
- [ ] ★ (по желанию): `ce_loss`, 5 запусков, кривые рядом с REINFORCE; ответ
- [ ] Код квиза по лекции 5 лежит в `quiz/week05.txt`
- [ ] Ноутбук выполнен целиком: «Restart & Run All» перед сдачей